In [4]:
# 문제 1. [문자열/구현] 설비 에러 로그 파싱

# 설명: 반도체 장비에서 발생한 로그 데이터가 문자열 배열로 주어집니다. 
# 각 로그는 "[시간] [장비ID] [상태코드]" 형식입니다. 
# 상태 코드가 "ERR_001"부터 "ERR_005" 사이인 로그만 추출하여, 각 장비ID별로 에러가 발생한 횟수를 내림차순으로 정렬하여 반환하는 함수를 작성하세요.


def parser_error_logs(logs):
    error_counts = {}
    target_errors = {"[ERR_001]", "[ERR_002]", "[ERR_003]", "[ERR_004]", "[ERR_005]"} # set 설정
    
    for log in logs:
        parts = log.strip().split(" ") # 공백 기준으로 나누기
        if len(parts) != 3:
            continue # 로그 형식이 잘못된 경우 무시 (예외처리)
        _, equipment_id, status_code = parts
        # 타겟 에러 코드에 해당하는 경우에만 카운트
        if status_code in target_errors:
            # get 메서드를 사용하여 장비ID별로 키가 없을 때의 기본갑을 0으로 설정하여 카운트 증가
            error_counts[equipment_id] = error_counts.get(equipment_id, 0) + 1
    
    
    # 1. 에러 발생 횟수 기준 내림차순 정렬
    # 2. 횟수가 같을 경우 장비ID 기준 오름차순 정렬
    sorted_equipment = sorted(error_counts.items(), key=lambda x: (-x[1], x[0]))
    return [eq[0] for eq in sorted_equipment] # 장비ID만 추출하여 반환
        
# 예시 입력
sample_logs = [
    "[10:00] [EQ_01] [ERR_002]",
    "[10:01] [EQ_02] [ERR_001]",
    "[10:05] [EQ_01] [ERR_002]",
    "[10:06] [EQ_03] [INFO_001]",
    "[10:10] [EQ_02] [ERR_005]"
]

print(parser_error_logs(sample_logs))

['[EQ_01]', '[EQ_02]']


In [5]:
# 문제 2. [시계열 전처리] 센서 데이터 결측치 처리 및 슬라이딩 윈도우

# 설명: 1초 단위로 측정된 온도 센서 데이터가 담긴 1차원 배열이 주어집니다. 센서 오류로 간혹 None 값이 포함되어 있습니다.

# None 값을 직전 값과 직후 값의 평균으로 선형 보간(Linear Interpolation)하세요. (처음이나 끝이 None인 경우는 없음)

# 보간된 데이터에서 Window Size = 5, Stride = 1인 슬라이딩 윈도우 배열을 생성하여 반환하세요.

# 핵심 포인트: 리스트 인덱싱을 활용한 주변값 탐색 및 시계열 데이터의 배치(Batch)화.

def preprocess_sensor_data(data):
    n = len(data)
    
    # 1. 결측치 처리(선형 보간)
    # 처음과 끝이 None이 아니라고 가정하므로, 1부터 n-1까지 탐색
    for i in range(1, n-1):
        if data[i] is None:
            # 직전 값과 직후 값의 평균으로 대체
            data[i] = (data[i-1] + data[i+2]) / 2
            
    # 2. 슬라이딩 윈도우 생성
    window_size = 5
    stride = 1
    result = []
    
    # 인덱스 범위 초과를 방지하기 위해 순회 범위 설정
    for i in range(0, n - window_size + 1 , stride):
        window = data[i:i+window_size]
        result.append(window)
    
    return result
        
# 예시 입력
sensor_data = [20.5, 21.0, None, 22.0, 22.5, None, 23.0, 23.5, 24.0]
print(preprocess_sensor_data(sensor_data))

[[20.5, 21.0, 21.75, 22.0, 22.5], [21.0, 21.75, 22.0, 22.5, 23.0], [21.75, 22.0, 22.5, 23.0, 23.0], [22.0, 22.5, 23.0, 23.0, 23.5], [22.5, 23.0, 23.0, 23.5, 24.0]]


In [2]:
# 문제 3. [Vision / PyTorch] Custom Dataset 구현
# 설명: data_dir 폴더 안에 normal/ 폴더와 defect/ 폴더가 있고, 
# 각각 이미지 파일(.jpg)들이 들어있습니다. 
# 이 디렉토리 구조를 읽어들여 이미지 파일 경로와 라벨(정상=0, 불량=1)을 매핑하는 PyTorch의 Dataset 클래스를 작성하세요.
# __getitem__ 메서드에서는 이미지를 불러와 $224 \times 224$로 리사이즈하고, 텐서로 변환하여 반환해야 합니다.

# 핵심 포인트: __init__, __len__, __getitem__ 3가지 필수 메서드의 역할 분리와 메모리 효율성(이미지는 __getitem__에서 로드).

import os
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class DefectDataset(Dataset):
    def __init__(self, data_dir):
        self.image_paths = []
        self.labels = []
        # 라벨 딕셔너리
        class_to_idx = {'normal' : 0, 'defect' : 1}
        
        # 디렉토리 순회하며 파일 경로 저장
        for class_name, label_idx in class_to_idx.items():
            class_dir = os.path.join(data_dir, class_name)
            # 해당 디렉토리가 존재하는지 확인
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append(os.path.join(class_dir, img_name))
                        self.labels.append(label_idx)
        # 텐서 변환 및 리사이징 정의
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB') # 이미지 로드 및 RGB 변환
        
        # 전처리
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        
        # 라벨을 PyTorch 텐서로 변환
        return image, torch.tensor(label, dtype=torch.long)


AttributeError: partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

In [ ]:
# 문제 4. [모델링] 전이 학습(Transfer Learning) 분류기 수정
# 핵심 포인트: torchvision 모델의 구조를 파악하고, 마지막 레이어(fc 또는 classifier)의 입출력 차원을 맞추는 방법.
import torch.nn as nn
import torchvision.models as models

def get_modified_resnet():
    # 1. ImageNet으로 사전 학습된 ResNet50 모델 불러오기
    model = models.resnet50(pretrained=True)
    # 모델의 특성 추출기 가중치를 동결하고 싶을 때
    # for param in model.parameters():
    #     param.requires_grad = False
    
    # 2. 기존 마지막 Fully Connected Layer(fc)의 입력 차원 확인
    num_ftrs = model.fc.in_features
    
    # 3. 우리가 원하는 클래스 개수(2개: 정상, 불량)로 마지막 레이어 교체
    model.fc = nn.Linear(num_ftrs, 2)
    
    return model

In [5]:
# 문제 5. [평가 지표] 불균형 데이터의 성능 평가 지표 계산

# 설명: 모델이 예측한 결과 배열 y_pred와 실제 정답 배열 y_true가 주어집니다. 
# (1: 불량, 0: 정상). 라이브러리(scikit-learn 등)를 사용하지 않고, 직접 Confusion Matrix의 요소(TP, TN, FP, FN)를 계산한 뒤,
# **정밀도(Precision)**와 재현율(Recall), 그리고 F1-score를 반환하는 함수를 작성하세요.


def calculate_metrics(y_true, y_pred):
    tp = 0  # True Positives
    tn = 0  # True Negatives
    fp = 0  # False Positives
    fn = 0  # False Negatives
    
    # 1. 요소 계산
    for true_val, pred_val in zip(y_true, y_pred):
        if true_val == 1 and pred_val == 1:
            tp += 1
        elif true_val == 0 and pred_val == 0:
            tn += 1
        elif true_val == 0 and pred_val == 1:
            fp += 1
        elif true_val == 1 and pred_val == 0:
            fn += 1
            
    # 2. 정밀도(Precision) 계산
    # 분모가 0이 되는 경우를 방지하기 위해 조건문 추가
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    
    # 3. 재현율(Recall) 계산
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # 4. F1-score 계산
    f1_score = 0.0
    if precision + recall > 0:
        f1_score = 2 * (precision * recall) / (precision + recall)
    
    return precision, recall, f1_score

# 예시 입력
y_true = [0, 1, 0, 0, 1, 1, 0]
y_pred = [0, 1, 1, 0, 0, 1, 0]
print(calculate_metrics(y_true, y_pred))


(0.6666666666666666, 0.6666666666666666, 0.6666666666666666)
